In [1]:
import sys, pandas, duckdb, pyarrow
print(sys.executable)
print(pandas.__version__, duckdb.__version__, pyarrow.__version__)

/Users/rajkandula/projects/walmart-demand-forecasting/.venv/bin/python
3.0.6 1.5.5 25.0.1


In [2]:
import sys
sys.path.append("..")
import pandas as pd
from src.config import RAW_DIR, PROCESSED_DIR

sales = pd.read_csv(RAW_DIR / "sales_train_evaluation.csv")
print(sales.shape)

def mb(df):
    return df.memory_usage(deep=True).sum() / 1e6

(30490, 1947)


In [3]:
sales["state_id"]

0        CA
1        CA
2        CA
3        CA
4        CA
         ..
30485    WI
30486    WI
30487    WI
30488    WI
30489    WI
Name: state_id, Length: 30490, dtype: str

In [4]:
sales["state_id"] == "TX"

0        False
1        False
2        False
3        False
4        False
         ...  
30485    False
30486    False
30487    False
30488    False
30489    False
Name: state_id, Length: 30490, dtype: bool

In [5]:
tx = sales[sales["state_id"] == "TX"] 
tx

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
12196,HOBBIES_1_001_TX_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,TX_1,TX,0,0,0,0,...,0,0,0,2,1,0,2,1,0,1
12197,HOBBIES_1_002_TX_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,TX_1,TX,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
12198,HOBBIES_1_003_TX_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,TX_1,TX,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12199,HOBBIES_1_004_TX_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,TX_1,TX,1,2,1,0,...,0,0,0,0,4,0,0,2,4,2
12200,HOBBIES_1_005_TX_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,TX_1,TX,0,0,0,0,...,0,0,1,0,3,1,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21338,FOODS_3_823_TX_3_evaluation,FOODS_3_823,FOODS_3,FOODS,TX_3,TX,0,0,0,0,...,1,0,0,0,1,0,1,1,0,1
21339,FOODS_3_824_TX_3_evaluation,FOODS_3_824,FOODS_3,FOODS,TX_3,TX,0,1,0,1,...,0,0,2,1,0,1,0,2,0,0
21340,FOODS_3_825_TX_3_evaluation,FOODS_3_825,FOODS_3,FOODS,TX_3,TX,0,0,0,2,...,3,1,1,2,0,2,5,0,0,3
21341,FOODS_3_826_TX_3_evaluation,FOODS_3_826,FOODS_3,FOODS,TX_3,TX,0,0,0,0,...,2,2,2,0,0,2,4,0,6,2


In [6]:
assert tx.shape == (9147, 1947)
print("Wide TX, default dtypes:", mb(tx), "MB")

Wide TX, default dtypes: 143.056716 MB


In [7]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
print("Wide TX, category ids:", mb(tx), "MB")


Wide TX, category ids: 143.056716 MB


In [8]:
tx[id_cols] = tx[id_cols].astype("category")
print("Wide TX, category ids:", mb(tx), "MB")
print(tx[id_cols].dtypes)

Wide TX, category ids: 142.50901 MB
id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
dtype: object


In [9]:
day_cols = [c for c in tx.columns if c.startswith("d_")]
print(len(day_cols))
tx[day_cols].sum()

1941


d_1        9438
d_2        9630
d_3        6778
d_4        7381
d_5        5912
          ...  
d_1937    10375
d_1938     9162
d_1939    12303
d_1940    13681
d_1941    14815
Length: 1941, dtype: int64

In [10]:
day_cols = [c for c in tx.columns if c.startswith("d_")]
print(len(day_cols))
wide_total = tx[day_cols].sum().sum()
print(wide_total)
print(tx[day_cols].max().max())

1941
19228405
634


In [11]:
tx_long = pd.melt(tx, id_vars=id_cols, var_name="d", value_name="sales")
print(tx_long.shape)
print(tx_long.dtypes)
print("Long, before dtype changes:", mb(tx_long), "MB")
print(tx_long.memory_usage(deep=True) / 1e6)

(17754327, 8)
id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
d                str
sales          int64
dtype: object
Long, before dtype changes: 525.12459 MB
Index         0.000132
id           35.845897
item_id      35.572285
dept_id      17.754445
cat_id       17.754373
store_id     17.754364
state_id     17.754338
d           240.654140
sales       142.034616
dtype: float64


In [12]:
tx_long["sales"] = tx_long["sales"].astype("int16")
tx_long["d"] = tx_long["d"].astype("category")
assert tx_long["sales"].sum() == wide_total
print(tx_long.dtypes)
print("Long, after dtype changes:", mb(tx_long), "MB")

id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
d           category
sales          int16
dtype: object
Long, after dtype changes: 213.479452 MB


In [13]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "tx_sales_long.parquet"
tx_long.to_parquet(out_path, index=False)
print("Parquet:", out_path.stat().st_size / 1e6, "MB")
print("CSV:", (RAW_DIR / "sales_train_evaluation.csv").stat().st_size / 1e6, "MB")

Parquet: 68.47012 MB
CSV: 121.736518 MB
